# Project distributions — material types & archetypes  *(Python)*
Python twin of `scripts/2026-06-15_distributions.R`. Distribution of each of the 7 material types (CSI bids vs the rest of the field) and project counts per archetype (field vs where CSI bids). Palette matches `utils/colors.py`.

In [ ]:
from pathlib import Path
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 50)

# --- absolute paths (edit BASE if you move the project) ----------------------
BASE = Path("/Users/t/Library/Mobile Documents/com~apple~CloudDocs/00 MSBA + Coding/"
            "00_summer 2026/00_CSI_ELP/01_Summer_2026")
OUT  = BASE / "02_modules/sweet_spot/outputs"
base = pd.read_csv(OUT / "2026-06-04_project_base.csv")
arch = pd.read_csv(OUT / "2026-06-04_project_archetypes.csv")[["Contract_Id", "rule_archetype"]]
df = base.merge(arch, on="Contract_Id", how="left")
print("merged:", df.shape, "| archetypes:", df["rule_archetype"].notna().sum())

In [ ]:
import seaborn as sns
ARCH = {"Reconstruct/Rebuild":"#7b3294","Mill & Overlay":"#fdae61","HMA Overlay/Resurface":"#2c7fb8",
        "Grading/Earthwork":"#d7191c","Aggregate/Base/Gravel":"#abdda4","Unknown/NoQty":"#bdbdbd","Other":"#999999"}
ORDER = list(ARCH)
df["bid_field"] = np.where(df["CSI_Participated"].astype(str).str.upper()=="TRUE","CSI bid","Rest of field")
MATS = {"MIX_TN_num":"MIX (asphalt,t)","GRADE_CY_num":"GRADE (earthwork,CY)","BASE_TN_num":"BASE (agg,t)",
        "SHLDR_TN_num":"SHLDR (shoulder,t)","MILL_SY_num":"MILL (milling,SY)","CRUSH_TN_num":"CRUSH (crushed,t)","STOCK_TN_num":"STOCK (stockpile,t)"}

### Material-quantity distributions — CSI bids vs the rest of the field (log10)

In [ ]:
long = df.melt(id_vars="bid_field", value_vars=list(MATS), var_name="material", value_name="qty")
long = long[long["qty"]>0].copy(); long["material"]=long["material"].map(MATS)
long["logq"]=np.log10(long["qty"])
g = sns.FacetGrid(long, col="material", col_wrap=2, hue="bid_field",
                  hue_order=["CSI bid","Rest of field"], palette={"CSI bid":"#1f4e79","Rest of field":"#bdbdbd"},
                  sharex=False, sharey=False, height=2.6, aspect=1.8)
g.map(sns.kdeplot, "logq", fill=True, alpha=.45).add_legend()
g.set_titles("{col_name}"); g.set_axis_labels("log10(quantity)","density")
plt.subplots_adjust(top=.93); g.figure.suptitle("Material distributions — CSI bids vs field", fontweight="bold"); plt.show()

### Projects by archetype — the whole field vs where CSI bids

In [ ]:
cnt = (df[df["rule_archetype"].notna()].groupby("rule_archetype")
       .agg(field=("Contract_Id","size"), bid=("bid_field", lambda s:(s=="CSI bid").sum())).reindex(ORDER).dropna())
ax = cnt.plot.bar(figsize=(10,5), color={"field":"#bdbdbd","bid":"#1f4e79"})
ax.set_title("Projects by archetype — field vs CSI bid"); ax.set_xlabel(""); ax.set_ylabel("projects")
for c in ax.containers: ax.bar_label(c, fontsize=8)
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()
print(cnt.astype(int).to_string())

### Material distributions colored by archetype

In [ ]:
la = df[df["rule_archetype"].notna()].melt(id_vars="rule_archetype", value_vars=list(MATS), var_name="material", value_name="qty")
la = la[la["qty"]>0].copy(); la["material"]=la["material"].map(MATS); la["logq"]=np.log10(la["qty"])
g2 = sns.FacetGrid(la, col="material", col_wrap=2, hue="rule_archetype", hue_order=ORDER, palette=ARCH,
                   sharex=False, sharey=False, height=2.6, aspect=1.8)
g2.map(sns.kdeplot, "logq", fill=True, alpha=.35).add_legend()
g2.set_titles("{col_name}"); g2.set_axis_labels("log10(quantity)","density"); plt.show()

## Takeaway
CSI's bids (blue) sit to the **right** of the field on MIX — it bids the larger asphalt jobs. Counts confirm it engages most on Large Volume & Complex/Messy. The interactive twins of these charts are on **dashboard page 8**; high-res ggplot versions in `outputs/`.